In [ ]:
import pandas as pd 
import re

package_data = pd.read_csv("../package_data.csv")

In [ ]:
def clean_version(tag_name):
    """Remove package name prefixes before version numbers."""
    tag_name = str(tag_name)
    
    # Strategy 1: Find @ followed by a proper version pattern (digit.digit...)
    # Handles: @org/package@1.0.0 -> 1.0.0
    matches = list(re.finditer(r'@(\d+\.\d+[^\s]*)$', tag_name))
    if matches:
        return matches[-1].group(1)
    
    # Strategy 2: Find / followed by version pattern (for package/version style)
    # Handles: package-name/v1.0.0 -> 1.0.0
    match = re.search(r'/[v]?(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 3: Find _ followed by v and version pattern
    # Handles: gmv3_v2.0.0 -> 2.0.0
    match = re.search(r'_v(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 4: Handle package-version style with hyphen
    # Handles: package-1.0.0 -> 1.0.0
    match = re.search(r'-(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 5: Fallback - remove any non-digit prefix (for v1.0.0 style)
    # Handles: v1.0.0 -> 1.0.0
    match = re.search(r'^[^\d]*(\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    return tag_name

In [ ]:
test_cases = [
    '12.0',
    'v3.0',
    'v2.0.0a0',
    '@0xsplits/splits-sdk@3.0.0',
    '@accelbyte/sdk-chat@6.3.0',
    '@2digits/constants@0.0.6',
    'vidgear-0.3.3',
    'resc-3.0.0',
    'dialtone-vue2/v2.112.0',
    'chacha20poly1305/1.6.2',
    'dialtone-vue3/v3.105.0-alpha.1',
    'arcor2/1.2.0',
    'arcor2_build/1.2.0',
    'gmv3_v2.0.0',
    'v10.0.0-chore_upgrade-deps',
    'rust_example_1_0_0'
]

print("Testing clean_version function:")
for test in test_cases:
    result = clean_version(test)
    print(f"{test:50} -> {result}")

In [ ]:
def is_semantic_version(tag_name):
    """
    Check if tag follows semantic versioning: MAJOR.MINOR.PATCH
    Also accepts minor versions (MAJOR.MINOR) and major versions (MAJOR)
    Allows suffixes like alpha, beta, rc, post, dev, etc.
    """
    tag_name = str(tag_name)
    
    # Pattern for semantic versioning with optional suffixes
    # Matches: X.Y.Z, X.Y, or X (where X, Y, Z are numbers)
    # Allows suffixes: -alpha, .post1, a0, b1, rc2, dev3, etc.
    pattern = r'^\d+(\.\d+)?(\.\d+)?([a-zA-Z0-9\.\-]+)?$'
    
    return bool(re.match(pattern, tag_name))

In [ ]:
# Apply the updated clean_version function
package_data['tag_name_test'] = package_data['tag_name'].apply(clean_version)

# Check semantic versioning
non_semver = package_data[~package_data['tag_name_test'].apply(is_semantic_version)]

print(f"Total tags: {len(package_data)}")
print(f"Number of non-semantic version tags: {len(non_semver)}")
print(f"Percentage: {len(non_semver)/len(package_data)*100:.2f}%")

# Show examples
print(f"\nExamples of remaining non-semantic version tags:")
print(non_semver['tag_name_test'])

In [ ]:
def is_basic_version_format(tag_name):
    """
    Check if tag follows basic version format:
    - number.number.number (e.g., 1.2.3)
    - number.number (e.g., 1.2)
    - number (e.g., 1)
    
    Does NOT allow any suffixes or prefixes.
    """
    tag_name = str(tag_name)
    
    # Strict pattern: ONLY digits and dots, nothing else
    # Must be: X, X.Y, or X.Y.Z where X, Y, Z are numbers
    pattern = r'^\d+(\.\d+)?(\.\d+)?$'
    
    return bool(re.match(pattern, tag_name))

# Find all tags that don't follow the basic format
non_basic_format = package_data[~package_data['tag_name_test'].apply(is_basic_version_format)]

print(f"Total tags: {len(package_data)}")
print(f"Tags NOT following X, X.Y, or X.Y.Z format: {len(non_basic_format)}")
print(f"Percentage: {len(non_basic_format)/len(package_data)*100:.2f}%")

# Show all unique non-basic format tags
print(f"\nAll non-basic format tags ({len(non_basic_format['tag_name_test'].unique())} unique):")
for tag in sorted(non_basic_format['tag_name_test'].unique()):
    print(tag)

In [ ]:
## delete these rows of data
package_data = package_data[package_data['tag_name_test'].apply(is_basic_version_format)]

## Now save the output!